In [17]:
#pip installs
%pip install numpy
%pip install pandas
%pip install gymnasium

In [18]:
#import
import numpy as np
import gymnasium as gym
from gym import Env, spaces

In [19]:
#skeleton
class CustomCliffWalker(Env):
  def __init__(self, rows=4, cols=12):
    super(CustomCliffWalker, self).__init__()

    #grid size -> default is 4*12 = 48
    self.rows = rows
    self.cols = cols

    #4 action spaces
    self.action_space = spaces.Discrete(4)

    #Observation space
    self.observation_space = spaces.Discrete(rows * cols)

    #cliff layout
    self.layout = np.zeros((rows, cols), dtype=int)

    #add the goal and cliff
    #goal
    self.layout[-1, -1] = 2

    #cliff
    for c in range(1, self.cols - 1):
      self.layout[-1, c] = 1

    #start
    self.start_pos = (self.rows - 1, 0)
    self.state = (0, -1)

  def step(self, action):
    x, y = self.state

    if action == 0:
      #left
      y = max(0, y - 1)
    elif action == 1:
      #down
      x = min(self.rows - 1, x + 1)
    elif action == 2:
      #right
      y = min(self.cols - 1, y + 1)
    elif action == 3:
      #up
      x = max(0, x - 1)


    self.state = (x, y)
    reward = -1
    if self.layout[x, y] == 1:
      #fell into cliff
      reward = -100
    elif self.layout[x, y] == 2:
      #reached goal
      reward = 0

    #termination check
    done = self.layout[x, y] == 2 or self.layout[x, y] == 1

    #return info
    return self._get_state_index(), reward, done, {}

  def reset(self):
    self.state = self.start_pos
    return self._get_state_index()


  def render(self):
    grid = np.array(self.layout, dtype=str)
    grid[self.layout == 0] = "."
    grid[self.layout == 1] = "C"
    grid[self.layout == 2] = "G"

    x, y = self.state

    #agent
    grid[x, y] = "A"

    print("\n".join(" ".join(row) for row in grid))

  #state encoding
  def _get_state_index(self):
    return self.state[0] * self.cols + self.state[1]

In [20]:
#run the env
env = CustomCliffWalker(rows=4, cols=12)
state = env.reset()

total_reward = 0
done = False

while not done:
  action = env.action_space.sample()
  next_state, reward, done, info = env.step(action)

  total_reward += reward

  action_names = ["Left", "Down", "Right", "Up"]
  print(f"\nAction: {action_names[action]}")

  env.render()
  print(f"Reward: {reward}")

  if reward == -100:
    print("Agent fell into the cliff!!")
  elif reward == 0:
    print("Agent reached the goal!!")


Action: Up
. . . . . . . . . . . .
. . . . . . . . . . . .
A . . . . . . . . . . .
. C C C C C C C C C C G
Reward: -1

Action: Left
. . . . . . . . . . . .
. . . . . . . . . . . .
A . . . . . . . . . . .
. C C C C C C C C C C G
Reward: -1

Action: Left
. . . . . . . . . . . .
. . . . . . . . . . . .
A . . . . . . . . . . .
. C C C C C C C C C C G
Reward: -1

Action: Left
. . . . . . . . . . . .
. . . . . . . . . . . .
A . . . . . . . . . . .
. C C C C C C C C C C G
Reward: -1

Action: Down
. . . . . . . . . . . .
. . . . . . . . . . . .
. . . . . . . . . . . .
A C C C C C C C C C C G
Reward: -1

Action: Right
. . . . . . . . . . . .
. . . . . . . . . . . .
. . . . . . . . . . . .
. A C C C C C C C C C G
Reward: -100
Agent fell into the cliff!!


In [21]:
#run the env
env = CustomCliffWalker(rows=4, cols=12)

#episodes
episodes = 5
for epi in range(episodes):
  state = env.reset()
  done = False
  total_reward = 0

  print(f"\n------Episode: {epi + 1} --------")
  env.render()

  while not done:
    action = env.action_space.sample()
    next_state, reward, done, info = env.step(action)

    total_reward += reward

    action_names = ["Left", "Down", "Right", "Up"]
    print(f"\nAction: {action_names[action]}")
    env.render()
    print(f"Reward: {reward}")

    if reward == -100:
      print("Agent fell into the cliff!!")
    elif reward == 0:
      print("Agent reached the goal!!")

  print(f"Episode Total Reward: {total_reward}")


------Episode: 1 --------
. . . . . . . . . . . .
. . . . . . . . . . . .
. . . . . . . . . . . .
A C C C C C C C C C C G

Action: Down
. . . . . . . . . . . .
. . . . . . . . . . . .
. . . . . . . . . . . .
A C C C C C C C C C C G
Reward: -1

Action: Up
. . . . . . . . . . . .
. . . . . . . . . . . .
A . . . . . . . . . . .
. C C C C C C C C C C G
Reward: -1

Action: Up
. . . . . . . . . . . .
A . . . . . . . . . . .
. . . . . . . . . . . .
. C C C C C C C C C C G
Reward: -1

Action: Left
. . . . . . . . . . . .
A . . . . . . . . . . .
. . . . . . . . . . . .
. C C C C C C C C C C G
Reward: -1

Action: Left
. . . . . . . . . . . .
A . . . . . . . . . . .
. . . . . . . . . . . .
. C C C C C C C C C C G
Reward: -1

Action: Up
A . . . . . . . . . . .
. . . . . . . . . . . .
. . . . . . . . . . . .
. C C C C C C C C C C G
Reward: -1

Action: Left
A . . . . . . . . . . .
. . . . . . . . . . . .
. . . . . . . . . . . .
. C C C C C C C C C C G
Reward: -1

Action: Down
. . . . . . . . . . . 